# Machine Learning Model Training

This notebook develops and evaluates machine learning models for wildfire burned-area prediction using the preprocessed Mesogeos wildfire dataset.

The workflow includes:

1. Loading the preprocessed training and testing datasets
2. Verifying feature consistency and data integrity
3. Establishing baseline models
4. Training and comparing multiple machine learning algorithms
5. Evaluating model performance using appropriate regression metrics
6. Investigating overfitting and generalisation performance
7. Hyperparameter tuning of selected models
8. Selecting the final predictive model
9. Evaluating the final model on the held-out test set
10. Saving the final model for integration into the prediction system

The target variable is the logarithmically transformed burned area:

`log(1 + burned_area_ha)`

The test set remains untouched during model development and hyperparameter tuning.

In [2]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# DEFINE PROJECT AND PROCESSED DATA PATHS

PROJECT_ROOT = Path.cwd().parents[1]

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"

print("Project root:")
print(PROJECT_ROOT)

print("\nProcessed data directory:")
print(PROCESSED_DIR)

Project root:
c:\Projects\ML Based Forest-Fire Prediction Project

Processed data directory:
c:\Projects\ML Based Forest-Fire Prediction Project\dataset\processed


In [4]:
# LOAD PROCESSED TRAINING AND TESTING DATA

X_train = pd.read_csv(PROCESSED_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DIR / "X_test.csv")

y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv").squeeze("columns")

print("Processed datasets loaded successfully.")

print("\nX_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("\ny_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

Processed datasets loaded successfully.

X_train shape: (9042, 44)
X_test shape : (2261, 44)

y_train shape: (9042,)
y_test shape : (2261,)


In [5]:
# LOAD FEATURE LIST

with open(PROCESSED_DIR / "final_features.json", "r") as f:
    final_features = json.load(f)

print("Number of final features:", len(final_features))

print("\nFinal features:")
for i, feature in enumerate(final_features, start=1):
    print(f"{i:02d}. {feature}")

Number of final features: 44

Final features:
01. latitude
02. longitude
03. elevation
04. slope_degrees
05. aspect
06. curvature
07. roads_distance_km
08. population
09. temperature_c
10. dew_point_c
11. relative_humidity
12. wind_speed
13. rainfall_mm
14. surface_pressure
15. solar_radiation
16. soil_moisture
17. ndvi
18. lai
19. lc_agriculture
20. lc_forest
21. lc_grassland
22. lc_settlement
23. lc_shrubland
24. lc_sparse_vegetation
25. lc_water_bodies
26. lc_wetland
27. year
28. month
29. day_of_year
30. month_sin
31. month_cos
32. day_of_year_sin
33. day_of_year_cos
34. temperature_c_missing
35. dew_point_c_missing
36. relative_humidity_missing
37. wind_speed_missing
38. rainfall_mm_missing
39. surface_pressure_missing
40. solar_radiation_missing
41. soil_moisture_missing
42. wind_direction_missing
43. wind_direction_sin
44. wind_direction_cos


In [6]:
assert list(X_train.columns) == final_features
assert list(X_test.columns) == final_features

print("Training feature order verified.")
print("Testing feature order verified.")
print("Feature consistency check passed.")

Training feature order verified.
Testing feature order verified.
Feature consistency check passed.


In [7]:
# CHECK FOR REMAINING MISSING VALUES

train_missing = X_train.isna().sum().sum()
test_missing = X_test.isna().sum().sum()

print("Missing values in X_train:", train_missing)
print("Missing values in X_test :", test_missing)

assert train_missing == 0
assert test_missing == 0

print("\nMissing-value validation passed.")

Missing values in X_train: 0
Missing values in X_test : 0

Missing-value validation passed.


In [9]:
# VERIFY TARGET VARIABLE

print("Training target statistics:")
display(y_train.describe())

print("\nTesting target statistics:")
display(y_test.describe())

print("\nTarget data types:")
print("y_train:", y_train.dtype)
print("y_test :", y_test.dtype)

Training target statistics:


count    9042.000000
mean        5.132169
std         1.201892
min         3.433987
25%         4.204693
50%         4.890349
75%         5.834079
max        11.586204
Name: log_burned_area, dtype: float64


Testing target statistics:


count    2261.000000
mean        5.084775
std         1.188039
min         3.433987
25%         4.158883
50%         4.844187
75%         5.768321
max        10.046678
Name: log_burned_area, dtype: float64


Target data types:
y_train: float64
y_test : float64


In [10]:
# VERIFY LOG-TRANSFORMED TARGET

print("Training target range:")
print("Minimum:", y_train.min())
print("Maximum:", y_train.max())

print("\nTesting target range:")
print("Minimum:", y_test.min())
print("Maximum:", y_test.max())

assert np.isfinite(y_train).all()
assert np.isfinite(y_test).all()

print("\nTarget validation passed.")

Training target range:
Minimum: 3.4339872044851463
Maximum: 11.586203807362043

Testing target range:
Minimum: 3.4339872044851463
Maximum: 10.046678392127044

Target validation passed.


In [11]:
# MODEL TRAINING DATA SUMMARY

print("=" * 60)
print("MODEL TRAINING DATA SUMMARY")
print("=" * 60)

print(f"Training records : {len(X_train):,}")
print(f"Testing records  : {len(X_test):,}")
print(f"Features         : {X_train.shape[1]}")
print(f"Target           : log(1 + burned_area_ha)")
print(f"Train/Test split : 80% / 20%")

print("\nData validation:")
print(f"Missing train values : {X_train.isna().sum().sum()}")
print(f"Missing test values  : {X_test.isna().sum().sum()}")
print(f"Feature consistency  : {list(X_train.columns) == list(X_test.columns)}")

print("\nAll initial model-training checks completed.")

MODEL TRAINING DATA SUMMARY
Training records : 9,042
Testing records  : 2,261
Features         : 44
Target           : log(1 + burned_area_ha)
Train/Test split : 80% / 20%

Data validation:
Missing train values : 0
Missing test values  : 0
Feature consistency  : True

All initial model-training checks completed.


# Baseline Model Selection

Following the preprocessing stage, several machine learning regression algorithms
will be trained and compared to establish baseline predictive performance.

The target variable is `log_burned_area`, which represents the logarithmically
transformed burned forest area. Since the target is continuous, regression
algorithms are used rather than classification algorithms.

The models selected for the baseline comparison represent different modelling
approaches, including linear regression, tree-based methods, ensemble learning,
boosting, and kernel-based regression.

## Models Selected for Evaluation

### 1. Linear Regression

Linear Regression will be used as a simple statistical baseline. It assumes that
the relationship between the predictor variables and burned area can be
approximated using a linear combination of the input features.

This provides a reference point against which more complex nonlinear models can
be evaluated.

### 2. Ridge Regression

Ridge Regression extends Linear Regression by applying L2 regularisation.
This helps reduce the influence of highly correlated features and can improve
model stability when many predictors are used.

### 3. Decision Tree Regressor

Decision Tree Regression will be evaluated as a nonlinear model capable of
capturing threshold-based relationships and interactions between environmental,
meteorological, geographical, and temporal features.

A single decision tree is also useful for understanding how a relatively simple
nonlinear model performs compared with ensemble methods.

### 4. Random Forest Regressor

Random Forest Regression combines multiple decision trees and averages their
predictions.

It is included because it can capture complex nonlinear relationships and
feature interactions while generally being more robust than a single decision
tree.

### 5. Extra Trees Regressor

Extra Trees Regression is another tree-based ensemble method that introduces
additional randomisation when constructing trees.

It will be compared with Random Forest to determine whether the increased
randomisation provides better generalisation on this dataset.

### 6. Gradient Boosting Regressor

Gradient Boosting builds an ensemble of decision trees sequentially, with each
new tree attempting to improve the errors made by previous trees.

This allows the model to capture complex nonlinear relationships that may not
be represented effectively by linear models.

### 7. HistGradientBoosting Regressor

HistGradientBoosting is an efficient histogram-based implementation of gradient
boosting.

It is included as a modern boosting approach and will be compared with the
traditional Gradient Boosting model.

### 8. Support Vector Regression (SVR)

Support Vector Regression will be evaluated as a kernel-based regression
approach.

SVR can model nonlinear relationships through kernel functions and provides a
useful comparison with tree-based and boosting algorithms.

Because SVR is sensitive to the scale of input variables, appropriate feature
standardisation will be applied before training.

## Model Comparison Strategy

All models will be trained using the same training dataset and evaluated using
the same held-out testing dataset.

The primary evaluation metrics will be:

- **R² (Coefficient of Determination):** measures how much variation in the
  target is explained by the model.
- **MAE (Mean Absolute Error):** measures the average absolute prediction error.
- **RMSE (Root Mean Squared Error):** gives greater weight to larger prediction
  errors.
- **Training R² vs Testing R²:** used to identify potential overfitting and
  evaluate model generalisation.

The baseline results will be used to identify the most promising models for
subsequent hyperparameter tuning.

The test dataset will not be used to select hyperparameters. It will remain
reserved for final model evaluation.

## Baseline Model Training

The selected regression algorithms are divided into scale-sensitive and
tree-based models.

Linear Regression, Ridge Regression, and Support Vector Regression require
feature standardisation because their performance is affected by differences
in feature magnitude.

Tree-based models do not require standardisation because their decision rules
are based on feature thresholds rather than distances or feature magnitude.

All models will be trained using the same training dataset and evaluated using
the same held-out testing dataset.

In [12]:
# IMPORT BASELINE MODELS

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)
from sklearn.svm import SVR

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print("Baseline model libraries imported successfully.")

Baseline model libraries imported successfully.


### Create the Models

In [13]:
# DEFINE BASELINE MODELS

scale_sensitive_models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),

    "Support Vector Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(
            kernel="rbf",
            C=1.0,
            epsilon=0.1
        ))
    ])
}


tree_models = {
    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        random_state=42
    )
}

print("Scale-sensitive models:")
for name in scale_sensitive_models:
    print("-", name)

print("\nTree-based models:")
for name in tree_models:
    print("-", name)

Scale-sensitive models:
- Linear Regression
- Ridge Regression
- Support Vector Regression

Tree-based models:
- Decision Tree
- Random Forest
- Extra Trees
- Gradient Boosting
- HistGradientBoosting


In [14]:
# MODEL EVALUATION FUNCTION

def evaluate_regression_model(model, X_train, y_train, X_test, y_test):
    """
    Train a regression model and calculate training/testing metrics.
    """

    # Train
    model.fit(X_train, y_train)

    # Predictions
    train_predictions = model.predict(X_train)
    test_predictions = model.predict(X_test)

    # Metrics
    train_r2 = r2_score(y_train, train_predictions)
    test_r2 = r2_score(y_test, test_predictions)

    train_mae = mean_absolute_error(y_train, train_predictions)
    test_mae = mean_absolute_error(y_test, test_predictions)

    train_rmse = np.sqrt(
        mean_squared_error(y_train, train_predictions)
    )

    test_rmse = np.sqrt(
        mean_squared_error(y_test, test_predictions)
    )

    return {
        "Train R²": train_r2,
        "Test R²": test_r2,
        "Train MAE": train_mae,
        "Test MAE": test_mae,
        "Train RMSE": train_rmse,
        "Test RMSE": test_rmse
    }

In [15]:
# TRAIN SCALE-SENSITIVE BASELINE MODELS

results = []

for name, model in scale_sensitive_models.items():

    print(f"Training {name}...")

    metrics = evaluate_regression_model(
        model,
        X_train,
        y_train,
        X_test,
        y_test
    )

    metrics["Model"] = name
    results.append(metrics)

print("\nScale-sensitive models completed.")

Training Linear Regression...
Training Ridge Regression...
Training Support Vector Regression...

Scale-sensitive models completed.


In [16]:
# TRAIN TREE-BASED BASELINE MODELS

for name, model in tree_models.items():

    print(f"Training {name}...")

    metrics = evaluate_regression_model(
        model,
        X_train,
        y_train,
        X_test,
        y_test
    )

    metrics["Model"] = name
    results.append(metrics)

print("\nTree-based models completed.")

Training Decision Tree...
Training Random Forest...
Training Extra Trees...
Training Gradient Boosting...
Training HistGradientBoosting...

Tree-based models completed.


In [17]:
# BASELINE MODEL COMPARISON

baseline_results = pd.DataFrame(results)

baseline_results = baseline_results[
    [
        "Model",
        "Train R²",
        "Test R²",
        "Train MAE",
        "Test MAE",
        "Train RMSE",
        "Test RMSE"
    ]
]

baseline_results = baseline_results.sort_values(
    by="Test R²",
    ascending=False
).reset_index(drop=True)

display(
    baseline_results.round(4)
)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,HistGradientBoosting,0.5401,0.2285,0.6506,0.8303,0.8150,1.0433
1,Random Forest,0.8877,0.2080,0.3173,0.8456,0.4027,1.0570
2,Extra Trees,1.0000,0.2051,0.0000,0.8475,0.0000,1.0590
3,Gradient Boosting,0.2722,0.2037,0.8128,0.8504,1.0253,1.0599
4,Support Vector Regression,0.2815,0.1847,0.7460,0.8349,1.0187,1.0725
5,Linear Regression,0.1306,0.1496,0.8832,0.8759,1.1206,1.0954
6,Ridge Regression,0.1306,0.1495,0.8832,0.8759,1.1206,1.0954
7,Decision Tree,1.0000,-0.6244,0.0000,1.1652,0.0000,1.5138


# Baseline Model Evaluation — Interpretation

The baseline experiments show substantial differences in predictive performance
and generalisation between the evaluated regression algorithms.

HistGradientBoosting achieved the strongest baseline performance, obtaining a
testing R² of approximately 0.229 and the lowest testing RMSE among the models.
It is therefore the leading baseline candidate.

Random Forest and Extra Trees also achieved testing R² values of approximately
0.208 and 0.205 respectively. However, both models showed substantial
differences between training and testing performance, indicating overfitting.
The Extra Trees model achieved a training R² of 1.00, demonstrating particularly
strong fitting to the training data.

Gradient Boosting achieved a testing R² of approximately 0.204 with a relatively
small training-testing performance gap, indicating more conservative
generalisation behaviour.

Support Vector Regression performed better than the linear models, suggesting
that nonlinear relationships are present in the dataset. Linear Regression and
Ridge Regression produced lower testing R² values of approximately 0.15 and
therefore primarily serve as reference baselines.

The Decision Tree model showed severe overfitting, achieving a training R² of
1.00 but a negative testing R² of approximately -0.62. This indicates poor
generalisation and makes the single decision tree a less promising candidate
for further development.

Based on the baseline comparison, HistGradientBoosting, Random Forest,
Extra Trees, and Gradient Boosting were selected as the primary candidates
for hyperparameter tuning.

The test dataset will continue to be treated as a held-out evaluation set
during model development.

# Hyperparameter Tuning

The baseline comparison identified HistGradientBoosting, Random Forest,
Extra Trees, and Gradient Boosting as the most promising models.

These models will now undergo hyperparameter tuning to investigate whether
their predictive performance and generalisation can be improved.

Hyperparameter tuning will be performed using cross-validation on the training
dataset only. The held-out testing dataset will not be used during the tuning
process.

The main objectives are:

- Improve predictive performance
- Reduce overfitting
- Improve generalisation to unseen observations
- Identify suitable model complexity
- Select the strongest candidate for final evaluation

In [18]:
# IMPORT HYPERPARAMETER TUNING TOOLS

from sklearn.model_selection import RandomizedSearchCV, KFold

print("Hyperparameter tuning libraries imported successfully.")

Hyperparameter tuning libraries imported successfully.


In [19]:
# CROSS-VALIDATION STRATEGY

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-fold cross-validation strategy created.")

5-fold cross-validation strategy created.


In [20]:
# HISTGRADIENTBOOSTING HYPERPARAMETER SEARCH

hist_model = HistGradientBoostingRegressor(
    random_state=42
)

hist_param_grid = {
    "learning_rate": [0.03, 0.05, 0.08, 0.1],
    "max_iter": [100, 200, 300, 400],
    "max_leaf_nodes": [15, 31, 63],
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [10, 20, 30, 50],
    "l2_regularization": [0.0, 0.1, 1.0, 5.0]
}

hist_search = RandomizedSearchCV(
    estimator=hist_model,
    param_distributions=hist_param_grid,
    n_iter=30,
    scoring="r2",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Starting HistGradientBoosting hyperparameter search...")

hist_search.fit(X_train, y_train)

print("\nHistGradientBoosting tuning completed.")

Starting HistGradientBoosting hyperparameter search...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

HistGradientBoosting tuning completed.


In [21]:
# BEST HISTGRADIENTBOOSTING PARAMETERS

print("Best HistGradientBoosting parameters:")
print(hist_search.best_params_)

print("\nBest cross-validation R²:")
print(round(hist_search.best_score_, 4))

Best HistGradientBoosting parameters:
{'min_samples_leaf': 30, 'max_leaf_nodes': 31, 'max_iter': 100, 'max_depth': 10, 'learning_rate': 0.05, 'l2_regularization': 1.0}

Best cross-validation R²:
0.1991


In [22]:
# EVALUATE TUNED HISTGRADIENTBOOSTING

best_hist_model = hist_search.best_estimator_

hist_train_predictions = best_hist_model.predict(X_train)
hist_test_predictions = best_hist_model.predict(X_test)

hist_tuned_results = {
    "Model": "Tuned HistGradientBoosting",
    "Train R²": r2_score(y_train, hist_train_predictions),
    "Test R²": r2_score(y_test, hist_test_predictions),
    "Train MAE": mean_absolute_error(y_train, hist_train_predictions),
    "Test MAE": mean_absolute_error(y_test, hist_test_predictions),
    "Train RMSE": np.sqrt(
        mean_squared_error(y_train, hist_train_predictions)
    ),
    "Test RMSE": np.sqrt(
        mean_squared_error(y_test, hist_test_predictions)
    )
}

display(
    pd.DataFrame([hist_tuned_results]).round(4)
)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,Tuned HistGradientBoosting,0.4049,0.2287,0.7398,0.8335,0.9271,1.0432


## Tuned HistGradientBoosting — Interpretation

Hyperparameter tuning was performed for the HistGradientBoosting model using
five-fold cross-validation on the training dataset.

The best configuration achieved a cross-validation R² of approximately 0.1991.
When evaluated on the held-out testing dataset, the tuned model achieved a
testing R² of 0.2287, compared with 0.2285 for the baseline model.

Therefore, hyperparameter tuning produced only a negligible improvement in
testing R².

The training R² decreased from 0.5401 for the baseline model to 0.4049 after
tuning. This indicates that the tuned model is less fitted to the training data
while maintaining almost identical testing performance.

The testing RMSE also changed only slightly, from 1.0433 to 1.0432, while the
testing MAE increased slightly from 0.8303 to 0.8335.

Overall, the tuning process did not substantially improve predictive
performance, but it produced a more conservative model with a smaller
training-testing performance gap. HistGradientBoosting therefore remains a
strong candidate, although additional models will be tuned before selecting
the final model.

## Random Forest Hyperparameter Tuning

The baseline Random Forest model achieved a testing R² of approximately 0.2080,
while its training R² was approximately 0.8877.

The large difference between training and testing performance indicates
substantial overfitting.

Hyperparameter tuning will therefore focus on controlling tree complexity and
improving generalisation. Parameters such as maximum tree depth, minimum
samples required for splitting and leaf nodes, and the number of trees will
be investigated.

The tuning process will use five-fold cross-validation on the training data
only. The held-out test dataset will remain untouched.

In [23]:
# RANDOM FOREST HYPERPARAMETER SEARCH

rf_model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

rf_param_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [None, 10, 15, 20, 25],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": [0.5, 0.7, 1.0]
}

rf_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=rf_param_grid,
    n_iter=30,
    scoring="r2",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Starting Random Forest hyperparameter search...")

rf_search.fit(X_train, y_train)

print("\nRandom Forest tuning completed.")

Starting Random Forest hyperparameter search...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Random Forest tuning completed.


In [24]:
print("Best Random Forest parameters:")
print(rf_search.best_params_)

print("\nBest cross-validation R²:")
print(round(rf_search.best_score_, 4))

Best Random Forest parameters:
{'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 0.7, 'max_depth': None}

Best cross-validation R²:
0.1903


In [26]:
# EVALUATE TUNED RANDOM FOREST

best_rf_model = rf_search.best_estimator_

rf_train_predictions = best_rf_model.predict(X_train)
rf_test_predictions = best_rf_model.predict(X_test)

rf_tuned_results = {
    "Model": "Tuned Random Forest",
    "Train R²": r2_score(y_train, rf_train_predictions),
    "Test R²": r2_score(y_test, rf_test_predictions),
    "Train MAE": mean_absolute_error(y_train, rf_train_predictions),
    "Test MAE": mean_absolute_error(y_test, rf_test_predictions),
    "Train RMSE": np.sqrt(
        mean_squared_error(y_train, rf_train_predictions)
    ),
    "Test RMSE": np.sqrt(
        mean_squared_error(y_test, rf_test_predictions)
    )
}

display(
    pd.DataFrame([rf_tuned_results]).round(4)
)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,Tuned Random Forest,0.8398,0.2177,0.366,0.8403,0.481,1.0506


## Tuned Random Forest — Interpretation

Hyperparameter tuning produced a modest improvement in Random Forest
performance.

The testing R² increased from 0.2080 for the baseline model to 0.2177 after
tuning. Testing MAE decreased from 0.8456 to 0.8403, while testing RMSE
decreased from 1.0570 to 1.0506.

The training R² decreased from 0.8877 to 0.8398, indicating that the tuned
model has reduced some of the overfitting observed in the baseline model.

Although the improvement is relatively small, the tuned Random Forest provides
better generalisation than the baseline configuration.

The tuned Random Forest currently remains slightly below the tuned
HistGradientBoosting model, which achieved a testing R² of 0.2287 and a testing
RMSE of 1.0432.

The Random Forest will therefore remain a candidate for final model selection,
while the remaining promising ensemble models are tuned and compared.

## Extra Trees Hyperparameter Tuning

The baseline Extra Trees model achieved a training R² of 1.00 while achieving
a testing R² of approximately 0.2051. This large difference indicates severe
overfitting.

Hyperparameter tuning will therefore focus on controlling model complexity
through parameters such as maximum tree depth, minimum samples required for
splitting and leaf nodes, feature selection, and the number of trees.

Five-fold cross-validation will be performed using the training dataset only.
The held-out testing dataset will remain untouched during hyperparameter
selection.

In [27]:
# EXTRA TREES HYPERPARAMETER SEARCH

extra_trees_model = ExtraTreesRegressor(
    random_state=42,
    n_jobs=-1
)

extra_trees_param_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [None, 10, 15, 20, 25],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": [0.5, 0.7, 1.0]
}

extra_trees_search = RandomizedSearchCV(
    estimator=extra_trees_model,
    param_distributions=extra_trees_param_grid,
    n_iter=30,
    scoring="r2",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Starting Extra Trees hyperparameter search...")

extra_trees_search.fit(X_train, y_train)

print("\nExtra Trees tuning completed.")

Starting Extra Trees hyperparameter search...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Extra Trees tuning completed.


In [28]:
# BEST EXTRA TREES PARAMETERS

print("Best Extra Trees parameters:")
print(extra_trees_search.best_params_)

print("\nBest cross-validation R²:")
print(round(extra_trees_search.best_score_, 4))

Best Extra Trees parameters:
{'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 1.0, 'max_depth': 25}

Best cross-validation R²:
0.1821


In [29]:
# EVALUATE TUNED EXTRA TREES

best_extra_trees_model = extra_trees_search.best_estimator_

extra_trees_train_predictions = (
    best_extra_trees_model.predict(X_train)
)

extra_trees_test_predictions = (
    best_extra_trees_model.predict(X_test)
)

extra_trees_tuned_results = {
    "Model": "Tuned Extra Trees",

    "Train R²": r2_score(
        y_train,
        extra_trees_train_predictions
    ),

    "Test R²": r2_score(
        y_test,
        extra_trees_test_predictions
    ),

    "Train MAE": mean_absolute_error(
        y_train,
        extra_trees_train_predictions
    ),

    "Test MAE": mean_absolute_error(
        y_test,
        extra_trees_test_predictions
    ),

    "Train RMSE": np.sqrt(
        mean_squared_error(
            y_train,
            extra_trees_train_predictions
        )
    ),

    "Test RMSE": np.sqrt(
        mean_squared_error(
            y_test,
            extra_trees_test_predictions
        )
    )
}

display(
    pd.DataFrame(
        [extra_trees_tuned_results]
    ).round(4)
)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,Tuned Extra Trees,0.9738,0.2117,0.1274,0.8413,0.1946,1.0546


## Tuned Extra Trees — Interpretation

The baseline Extra Trees model showed severe overfitting, with a training R² of
1.0000 compared with a testing R² of 0.2051. Hyperparameter tuning was
therefore performed to reduce model complexity and improve generalisation.

The best configuration selected by five-fold cross-validation was:

- `n_estimators = 500`
- `min_samples_split = 2`
- `min_samples_leaf = 2`
- `max_features = 1.0`
- `max_depth = 25`

The tuned model achieved a cross-validation R² of 0.1821 and a testing R² of
0.2117. This represents a small improvement over the baseline testing R² of
0.2051.

The training R² decreased from 1.0000 to 0.9738, indicating that restricting
the maximum tree depth reduced the extreme overfitting observed in the
baseline model. However, the remaining difference between training and
testing performance indicates that the model still fits the training data
considerably more closely than the unseen data.

The tuned model achieved a testing MAE of 0.8413 and a testing RMSE of 1.0546.
These results remain slightly weaker than those obtained by the current
best-performing HistGradientBoosting model.

Overall, hyperparameter tuning produced a modest improvement in Extra Trees
performance but did not substantially improve predictive capability. The
tuned Extra Trees model will therefore be retained for the final model
comparison, while the remaining candidate models are evaluated.

## Gradient Boosting Hyperparameter Tuning

The baseline Gradient Boosting model achieved a testing R² of approximately
0.2037. Hyperparameter tuning will be performed to investigate whether the
model can achieve improved generalisation on unseen data.

The tuning process will focus on the number of boosting stages, learning rate,
tree depth, minimum samples per leaf, and subsampling. These parameters control
the complexity and learning behaviour of the boosting model.

Five-fold cross-validation will be performed using the training dataset only.
The held-out testing dataset will remain untouched during hyperparameter
selection.

In [30]:
# GRADIENT BOOSTING HYPERPARAMETER SEARCH

gradient_boosting_model = GradientBoostingRegressor(
    random_state=42
)

gradient_boosting_param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4, 5],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "subsample": [0.7, 0.8, 1.0]
}

gradient_boosting_search = RandomizedSearchCV(
    estimator=gradient_boosting_model,
    param_distributions=gradient_boosting_param_grid,
    n_iter=30,
    scoring="r2",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Starting Gradient Boosting hyperparameter search...")

gradient_boosting_search.fit(X_train, y_train)

print("\nGradient Boosting tuning completed.")

Starting Gradient Boosting hyperparameter search...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Gradient Boosting tuning completed.


In [31]:
# BEST GRADIENT BOOSTING PARAMETERS

print("Best Gradient Boosting parameters:")
print(gradient_boosting_search.best_params_)

print("\nBest cross-validation R²:")
print(round(gradient_boosting_search.best_score_, 4))

Best Gradient Boosting parameters:
{'subsample': 0.7, 'n_estimators': 300, 'min_samples_split': 20, 'min_samples_leaf': 5, 'max_depth': 5, 'learning_rate': 0.03}

Best cross-validation R²:
0.197


In [32]:
# EVALUATE TUNED GRADIENT BOOSTING

best_gradient_boosting_model = (
    gradient_boosting_search.best_estimator_
)

gb_train_predictions = (
    best_gradient_boosting_model.predict(X_train)
)

gb_test_predictions = (
    best_gradient_boosting_model.predict(X_test)
)

gradient_boosting_tuned_results = {
    "Model": "Tuned Gradient Boosting",

    "Train R²": r2_score(
        y_train,
        gb_train_predictions
    ),

    "Test R²": r2_score(
        y_test,
        gb_test_predictions
    ),

    "Train MAE": mean_absolute_error(
        y_train,
        gb_train_predictions
    ),

    "Test MAE": mean_absolute_error(
        y_test,
        gb_test_predictions
    ),

    "Train RMSE": np.sqrt(
        mean_squared_error(
            y_train,
            gb_train_predictions
        )
    ),

    "Test RMSE": np.sqrt(
        mean_squared_error(
            y_test,
            gb_test_predictions
        )
    )
}

display(
    pd.DataFrame(
        [gradient_boosting_tuned_results]
    ).round(4)
)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,Tuned Gradient Boosting,0.4425,0.2222,0.7137,0.8364,0.8973,1.0476


## Tuned Gradient Boosting — Interpretation

Hyperparameter tuning was performed for the Gradient Boosting regression
model using five-fold cross-validation on the training dataset.

The best configuration was:

- `subsample = 0.7`
- `n_estimators = 300`
- `min_samples_split = 20`
- `min_samples_leaf = 5`
- `max_depth = 5`
- `learning_rate = 0.03`

The tuned model achieved a cross-validation R² of 0.1970 and a testing R² of
0.2222. This represents an improvement over the baseline Gradient Boosting
testing R² of 0.2037.

The model achieved a testing MAE of 0.8364 and a testing RMSE of 1.0476.
The training R² was 0.4425, compared with a testing R² of 0.2222. Although
some difference remains between training and testing performance, the model
does not exhibit the extreme overfitting observed in the baseline Extra Trees
and Random Forest models.

The tuned Gradient Boosting model therefore provides competitive predictive
performance and demonstrates reasonable generalisation to unseen data.

However, its testing R² of 0.2222 remains slightly below the tuned
HistGradientBoosting model, which currently achieves the best testing R² of
0.2287 and the lowest testing RMSE of 1.0432.

The tuned Gradient Boosting model will therefore be retained for the final
model comparison.

## Final Model Comparison

All baseline and tuned regression models are compared using the same held-out
testing dataset.

The primary evaluation metric is the testing R² score because it measures how
much variation in the transformed burned-area target is explained by the model.
Testing MAE and RMSE are also considered because they provide additional
information about prediction error.

Training R² is included to assess potential overfitting by comparing training
and testing performance.

The test dataset is not used during hyperparameter optimisation. Therefore,
the final comparison provides an independent evaluation of model
generalisation.

In [33]:
# FINAL MODEL COMPARISON

final_results = [
    {
        "Model": "Linear Regression",
        "Train R²": 0.1306,
        "Test R²": 0.1496,
        "Train MAE": 0.8832,
        "Test MAE": 0.8759,
        "Train RMSE": 1.1206,
        "Test RMSE": 1.0954
    },
    {
        "Model": "Ridge Regression",
        "Train R²": 0.1306,
        "Test R²": 0.1495,
        "Train MAE": 0.8832,
        "Test MAE": 0.8759,
        "Train RMSE": 1.1206,
        "Test RMSE": 1.0954
    },
    {
        "Model": "Support Vector Regression",
        "Train R²": 0.2815,
        "Test R²": 0.1847,
        "Train MAE": 0.7460,
        "Test MAE": 0.8349,
        "Train RMSE": 1.0187,
        "Test RMSE": 1.0725
    },
    {
        "Model": "Decision Tree",
        "Train R²": 1.0000,
        "Test R²": -0.6244,
        "Train MAE": 0.0000,
        "Test MAE": 1.1652,
        "Train RMSE": 0.0000,
        "Test RMSE": 1.5138
    },
    {
        "Model": "Random Forest",
        "Train R²": 0.8877,
        "Test R²": 0.2080,
        "Train MAE": 0.3173,
        "Test MAE": 0.8456,
        "Train RMSE": 0.4027,
        "Test RMSE": 1.0570
    },
    {
        "Model": "Tuned Random Forest",
        "Train R²": 0.8398,
        "Test R²": 0.2177,
        "Train MAE": 0.3660,
        "Test MAE": 0.8403,
        "Train RMSE": 0.4810,
        "Test RMSE": 1.0506
    },
    {
        "Model": "Extra Trees",
        "Train R²": 1.0000,
        "Test R²": 0.2051,
        "Train MAE": 0.0000,
        "Test MAE": 0.8475,
        "Train RMSE": 0.0000,
        "Test RMSE": 1.0590
    },
    {
        "Model": "Tuned Extra Trees",
        "Train R²": 0.9738,
        "Test R²": 0.2117,
        "Train MAE": 0.1274,
        "Test MAE": 0.8413,
        "Train RMSE": 0.1946,
        "Test RMSE": 1.0546
    },
    {
        "Model": "Gradient Boosting",
        "Train R²": 0.2722,
        "Test R²": 0.2037,
        "Train MAE": 0.8128,
        "Test MAE": 0.8504,
        "Train RMSE": 1.0253,
        "Test RMSE": 1.0599
    },
    {
        "Model": "Tuned Gradient Boosting",
        "Train R²": 0.4425,
        "Test R²": 0.2222,
        "Train MAE": 0.7137,
        "Test MAE": 0.8364,
        "Train RMSE": 0.8973,
        "Test RMSE": 1.0476
    },
    {
        "Model": "HistGradientBoosting",
        "Train R²": 0.5401,
        "Test R²": 0.2285,
        "Train MAE": 0.6506,
        "Test MAE": 0.8303,
        "Train RMSE": 0.8150,
        "Test RMSE": 1.0433
    },
    {
        "Model": "Tuned HistGradientBoosting",
        "Train R²": 0.4049,
        "Test R²": 0.2287,
        "Train MAE": 0.7398,
        "Test MAE": 0.8335,
        "Train RMSE": 0.9271,
        "Test RMSE": 1.0432
    }
]

final_results_df = pd.DataFrame(final_results)

final_results_df = final_results_df.sort_values(
    by="Test R²",
    ascending=False
).reset_index(drop=True)

display(
    final_results_df.round(4)
)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,Tuned HistGradientBoosting,0.4049,0.2287,0.7398,0.8335,0.9271,1.0432
1,HistGradientBoosting,0.5401,0.2285,0.6506,0.8303,0.8150,1.0433
2,Tuned Gradient Boosting,0.4425,0.2222,0.7137,0.8364,0.8973,1.0476
3,Tuned Random Forest,0.8398,0.2177,0.3660,0.8403,0.4810,1.0506
4,Tuned Extra Trees,0.9738,0.2117,0.1274,0.8413,0.1946,1.0546
5,Random Forest,0.8877,0.2080,0.3173,0.8456,0.4027,1.0570
6,Extra Trees,1.0000,0.2051,0.0000,0.8475,0.0000,1.0590
7,Gradient Boosting,0.2722,0.2037,0.8128,0.8504,1.0253,1.0599
8,Support Vector Regression,0.2815,0.1847,0.7460,0.8349,1.0187,1.0725
9,Linear Regression,0.1306,0.1496,0.8832,0.8759,1.1206,1.0954


In [34]:
# FINAL MODEL SELECTION

best_model_name = final_results_df.loc[0, "Model"]
best_test_r2 = final_results_df.loc[0, "Test R²"]
best_test_rmse = final_results_df.loc[0, "Test RMSE"]
best_test_mae = final_results_df.loc[0, "Test MAE"]

print("FINAL MODEL SELECTION")
print("=" * 50)

print(f"Best model      : {best_model_name}")
print(f"Testing R²      : {best_test_r2:.4f}")
print(f"Testing MAE     : {best_test_mae:.4f}")
print(f"Testing RMSE    : {best_test_rmse:.4f}")

FINAL MODEL SELECTION
Best model      : Tuned HistGradientBoosting
Testing R²      : 0.2287
Testing MAE     : 0.8335
Testing RMSE    : 1.0432


## Final Model Selection — Interpretation

A total of twelve baseline and tuned regression configurations were evaluated
using the same held-out testing dataset.

The tuned HistGradientBoosting model achieved the highest testing R² of
0.2287. It also achieved the lowest testing RMSE of 1.0432 and the lowest
testing MAE of 0.8335 among the evaluated models.

The model therefore provides the strongest overall predictive performance on
unseen data.

The improvement over the baseline HistGradientBoosting model was modest,
with testing R² increasing from 0.2285 to 0.2287. However, the tuned model
maintained competitive testing error while reducing the training R² from
0.5401 to 0.4049, indicating reduced model complexity and a smaller
training-testing performance gap.

The final model was selected based on testing performance rather than
training performance. This is particularly important because several
tree-based models achieved very high training R² values while producing
considerably lower testing R² values, indicating overfitting.

The tuned HistGradientBoosting model was therefore selected as the final
candidate for the forest-fire burned-area prediction system.